# DeepDream minimal (FRA - Aula 23)

Este notebook implementa um exemplo mínimo de **DeepDream** usando uma imagem de um felino da Wikipedia e a arquitetura mostrada na aula (InceptionV3). Ele roda no **Google Colab**.

**Imagem original (para display.html):**
https://commons.wikimedia.org/wiki/File:Felis_catus-cat_on_snow.jpg


In [ ]:
# imports básicos
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.applications import inception_v3

print(tf.__version__)


In [ ]:
# URL da imagem (conforme instrução)
url = "https://commons.wikimedia.org/wiki/Special:FilePath/Felis_catus-cat_on_snow.jpg"


In [ ]:
# download e leitura da imagem
import PIL.Image as PILImage
import requests
from io import BytesIO

response = requests.get(url)
img = PILImage.open(BytesIO(response.content))
img


In [ ]:
# utilitários de pré-processamento

def preprocess(img):
    img = np.array(img)
    img = inception_v3.preprocess_input(img)
    img = tf.convert_to_tensor(img)
    return img

def deprocess(img):
    img = 255 * (img + 1.0) / 2.0
    return tf.cast(img, tf.uint8)


In [ ]:
# carrega modelo base
base_model = inception_v3.InceptionV3(include_top=False, weights='imagenet')

# camadas alvo (podem ser ajustadas)
layer_names = [
    'mixed3',
    'mixed5',
]

layers = [base_model.get_layer(name).output for name in layer_names]

# cria modelo de ativação
feature_extractor = tf.keras.Model(inputs=base_model.input, outputs=layers)


In [ ]:
# define função de sonho

def calc_loss(img, model):
    img_batch = tf.expand_dims(img, axis=0)
    layer_activations = model(img_batch)
    if len(layer_activations) == 1:
        layer_activations = [layer_activations]
    losses = []
    for act in layer_activations:
        losses.append(tf.reduce_mean(act))
    return tf.reduce_sum(losses)

@tf.function
def dream_step(img, model, step_size):
    with tf.GradientTape() as tape:
        tape.watch(img)
        loss = calc_loss(img, model)
    gradients = tape.gradient(loss, img)
    gradients /= tf.math.reduce_std(gradients) + 1e-8
    img = img + gradients * step_size
    img = tf.clip_by_value(img, -1.0, 1.0)
    return img


In [ ]:
# Main Loop (sem oitavas)

def run_deep_dream_simple(img, steps=50, step_size=0.01):
    img = preprocess(img)
    for step in range(steps):
        img = dream_step(img, feature_extractor, step_size)
    return deprocess(img)

result_simple = run_deep_dream_simple(img, steps=50, step_size=0.01)
plt.figure(figsize=(10, 10))
plt.imshow(result_simple)
plt.axis('off')
plt.title('DeepDream - Main Loop')


In [ ]:
# DeepDream com oitavas

def run_deep_dream_with_octaves(img, steps=50, step_size=0.01, octaves=1, octave_scale=1.4):
    img = preprocess(img)
    base_shape = tf.shape(img)[:-1]
    float_base_shape = tf.cast(base_shape, tf.float32)

    for octave in range(octaves):
        new_shape = tf.cast(float_base_shape * (octave_scale ** octave), tf.int32)
        img = tf.image.resize(img, new_shape)

        for step in range(steps):
            img = dream_step(img, feature_extractor, step_size)

    return deprocess(img)

result_octave = run_deep_dream_with_octaves(img, steps=50, step_size=0.01, octaves=2, octave_scale=1.4)
plt.figure(figsize=(10, 10))
plt.imshow(result_octave)
plt.axis('off')
plt.title('DeepDream - até uma oitava (2 níveis)')


## Explicação dos resultados

- **Imagem onírica obtida por Main Loop:**
  O *Main Loop* aplica apenas o gradiente em uma única escala da imagem. O resultado tende a realçar padrões locais já presentes (texturas de pelo e neve), adicionando formas repetitivas onde o modelo encontra ativação alta, mas sem alterar a estrutura global da imagem.

- **Imagem onírica obtida ao levar o modelo até uma oitava:**
  Ao usar oitavas, o processo ocorre em múltiplas escalas (imagem reamostrada). Isso faz surgir padrões maiores e mais organizados, pois o modelo “enxerga” a imagem em tamanhos diferentes e amplifica estruturas em níveis mais amplos.

- **Diferenças entre imagens oníricas obtidas com Main Loop e levando o modelo até a oitava:**
  O *Main Loop* realça principalmente detalhes pequenos (efeito mais sutil). Já o uso de oitavas introduz elementos em diferentes escalas, tornando o efeito visual mais forte e com padrões que influenciam áreas maiores da imagem.
